## Extraction features for use in a Self Organizing Map 

Dataset from CLAS: A Database for Cognitive Load, Affect and Stress Recognition”  
Using library Neurokit2

### 1. Imports used during the project 

In [1]:
# Cell 1 
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm
import time
import matplotlib.pyplot as plt

from scipy.signal import resample_poly

### 2. Windowing & Samplerate 

Sliding windows 60 sec with 15 sec overlap

In [2]:
# Cell 2 

# Sampling frequency (Hz)
FS = 200      

# Sliding window length (seconds)
WINDOW_SEC = 60          

WINDOW_SAMPLES = FS * WINDOW_SEC

# Step size (seconds)
STEP_SEC = 15               


### 3 Resample signal function 

Using polyphase filtering to fit the sensors on ESP32-S3

In [3]:
# Cell 3

# Resample the signal to the desired sampling frequency
def resample_signal(signal, fs_in=256, fs_out=200):

    return resample_poly(signal, fs_out, fs_in)

In [4]:
import warnings
warnings.filterwarnings("ignore")

### 4. Extract PPG signal

Using Neurokit2 library for peak detection.

- elgendi
- charlton 
- centriod 

*Output*: RR (IBI) interval analysis, HR, HRV_RMSSD

In [5]:
# Cell 4

def extract_features_ppg(df, participant_id):

    FS = 200
    
    # Load and clean     
    ppg = pd.to_numeric(df["ppg"], errors="coerce")   
    ppg = ppg.interpolate().bfill().ffill()

    
    # RESAMPLE   
    ppg = resample_poly(ppg.values, FS, 256)

    
    # Trim edges to remove artifacts   
    ppg = ppg[FS * 3:-FS * 3]
   
    # PPG PROCESSING with NeuroKit2. Specially using Elgendi and charlton for better peak detection.
    signals_ppg, info_ppg = nk.ppg_process(
        ppg,
        sampling_rate=FS,
        method="elgendi",
        method_peaks="charlton"
    )

    signal_clean = signals_ppg["PPG_Clean"].values
    rpeaks = info_ppg["PPG_Peaks"]

    # safety check: if we have too few peaks, skip this participant
    if len(rpeaks) < 30:
        return pd.DataFrame()  

    
    # Centriod refinement of peaks to improve RR interval accuracy.
    def refine_peaks_centroid(signal, peaks, fs, window=0.05):
        refined = []
        w = int(window * fs)

        for p in peaks:
            start = max(0, p - w)
            end = min(len(signal), p + w)

            segment = signal[start:end]
            x = np.arange(start, end)

            y = segment - np.min(segment)

            if np.sum(y) == 0 or len(segment) == 0:
                refined.append(p)
                continue

            centroid = np.sum(x * y) / np.sum(y)
            refined.append(int(centroid))

        return np.array(refined)

    rpeaks = refine_peaks_centroid(signal_clean, rpeaks, FS)

    
    # RR INTERVALS
    rr_intervals = np.diff(rpeaks) / FS * 1000  
    rr_times_sec = (rpeaks[1:] + rpeaks[:-1]) / 2 / FS

    features = []

    step_samples = FS * STEP_SEC
    signal_len = len(signal_clean)

  
    # SLIDING WINDOWS    
    for start in range(0, signal_len - WINDOW_SAMPLES + 1, step_samples):

        end = start + WINDOW_SAMPLES
        window = {}

        start_sec = start / FS
        end_sec = end / FS      

        # RR SELECTION       
        rr_mask = (rr_times_sec >= start_sec) & (rr_times_sec < end_sec)
        rr_win = rr_intervals[rr_mask]

        # RR FILTERING
        rr_win = rr_win[(rr_win > 400) & (rr_win < 1200)]
        
        if len(rr_win) >= 30:

            window ["HR"] = 60000 / np.mean(rr_win)

            # Diff RR and quality check
            diff_rr = np.diff(rr_win)
            valid_diff = np.abs(diff_rr) < 150
            diff_rr = diff_rr[valid_diff]

            rr_quality = np.sum(valid_diff) / len(valid_diff) if len(valid_diff) > 0 else 0

            
            # HRV
            if len(diff_rr) > 0:
                window["HRV_RMSSD"] = np.sqrt(np.mean(diff_rr ** 2))
                window["RR_diff_std"] = np.std(diff_rr)
            else:
                window["HRV_RMSSD"] = np.nan
                window["RR_diff_std"] = np.nan

            window["HRV_SDNN"] = np.std(rr_win, ddof=1)

            # -------------------------
            # RR DIAGNOSTICS
            # -------------------------
            window["RR_mean"] = np.mean(rr_win)
            window["RR_std"] = np.std(rr_win)
            window["RR_min"] = np.min(rr_win)
            window["RR_max"] = np.max(rr_win)
            window["RR_count"] = len(rr_win)
            window["RR_quality"] = rr_quality
            window["RR_valid"] = 1

        else:
            window["HRV_RMSSD"] = np.nan
            window["HRV_SDNN"] = np.nan
            window["RR_mean"] = np.nan
            window["RR_std"] = np.nan
            window["RR_min"] = np.nan
            window["RR_max"] = np.nan
            window["RR_count"] = 0
            window["RR_diff_std"] = np.nan
            window["RR_quality"] = 0
            window["RR_valid"] = 0

        # -------------------------
        # METADATA
        # -------------------------
        window["participant"] = participant_id
        window["time_sec"] = start // FS

        features.append(window)

    return pd.DataFrame(features)

### 5. Extract features from GSR

Using Neurokit2 

- seperate tonic/phasic
- peak detection 

*Output*: SC_PH, SC_RR, EDA_SLOPE, 

In [10]:
def extract_features_GSR(df, participant_id, return_signals=False):

    RAW_FS = 256
    TARGET_FS = 200
    FS = TARGET_FS

    # Load and clean EDA signal
    eda = pd.to_numeric(df["gsr"], errors="coerce")
    eda = eda.interpolate().bfill().ffill()

    low, high = np.percentile(eda, [1, 99])
    eda = np.clip(eda, low, high)

   

    # Resample EDA signal to fit ESP32S3
    eda = resample_poly(eda.values, TARGET_FS, RAW_FS)

    # Trim edges to remove artifacts
    trim = FS * 5
    eda = eda[trim:-trim]

    eda_clean = nk.eda_clean(eda, sampling_rate=FS, method="neurokit")

    eda_decomp = nk.eda_phasic(eda_clean, sampling_rate=FS, method="highpass")

    tonic = eda_decomp["EDA_Tonic"].values
    phasic = eda_decomp["EDA_Phasic"].values

    phasic = np.maximum(phasic, 0)
    
    print("Phasic stats:")
    print("min:", np.min(phasic))
    print("max:", np.max(phasic))
    print("std:", np.std(phasic))

    amp_threshold = 0.05

    try: 

        signals_scr, _ = nk.eda_peaks(
            phasic,
            sampling_rate=FS,
            method="neurokit",
            amplitude_min=amp_threshold
        )

        scr = signals_scr["SCR_Peaks"].values

        print(f"[{participant_id}] Phasic peaks detected:", np.sum(scr))

        print(f"[{participant_id}] Threshold:", amp_threshold)
        print(f"[{participant_id}] SCR count:", np.sum(scr)) 

        min_distance_sec = 1.0
        min_samples = int(FS * min_distance_sec)

        filtered_scr = np.zeros_like(scr)
        last_peak = -min_samples

        for i in np.where(scr == 1)[0]:
            if i - last_peak >= min_samples:
                filtered_scr[i] = 1
                last_peak = i

        scr = filtered_scr

        print(f"[{participant_id}] Filtered SCR count:", np.sum(scr))



        if np.sum(scr) == 0:
            print(f"[{participant_id}] No SCR peaks found")
            # Set to zero to avoid issues in feature extraction
            scr = np.zeros_like(scr) 
        # Functions to extract features from sliding windows
    except Exception as e:
        print(f"[{participant_id}] EDA peak detection failed:", e)

        scr = np.zeros_like(phasic)
    
    W = FS * WINDOW_SEC
    S = FS * STEP_SEC

    features = []

    for start in range(0, len(tonic) - W + 1, S):

        end = start + W

        tonic_win = tonic[start:end]
        phasic_win = phasic[start:end]
        scr_win = scr[start:end]

        # Clip outliers. 
        phasic_cap = np.percentile(phasic_win, 99)
        phasic_win_clipped = np.clip(phasic_win, 0, phasic_cap)

        # SC_PH: Mean squared phasic activity
        sc_ph = np.mean(phasic_win_clipped ** 2)

        # SC_RR: SCR rate (number of SCRs per second)
        scr_count = np.sum(scr_win == 1)
        sc_rr = scr_count / WINDOW_SEC
        
        # EDA_SLOPE: Slope of tonic component (optinal to SOM)
        tonic_norm = (tonic_win - np.mean(tonic_win)) / (np.std(tonic_win) + 1e-8)
        x = np.arange(len(tonic_norm))
        slope = np.polyfit(x, tonic_norm, 1)[0]

        features.append({
            "SC_PH": sc_ph,
            "SC_RR": sc_rr,
            "EDA_slope": slope,
            "participant": participant_id,
            "time_sec": start // FS
        })

    features_df = pd.DataFrame(features)

    # Debug data with signals
    if return_signals:
        debug = pd.DataFrame({
            "eda_raw": eda,
            "eda_clean": eda_clean,
            "tonic": tonic,
            "phasic": phasic,
            "scr": scr
        })
        return features_df, debug

    return features_df

### 6. Function for extract PPG and EDA and merge togheter 

In [ ]:
# Cell 6

def extract_features_eda_ppg(df, participant_id):

    
    # Extract EDA features
    eda_features = extract_features_GSR(df, participant_id)

    if eda_features is None or len(eda_features) == 0:
        return None

   
    # Extract PPG features
    ppg_features = extract_features_ppg(df, participant_id)

    if ppg_features is None or len(ppg_features) == 0:
        return None

    
    # MERGE     
    merged = pd.merge(
        ppg_features,
        eda_features,
        on=["participant", "time_sec"],
        how="inner"
    )

    return merged

### 7. Function to get files for each participant in dataset

In [ ]:
# cell 7 

def get_participant_files(data_dir, start=1, end=60):

    files = sorted(
        data_dir.rglob("full_gsr_ppg*.csv"),
        key=lambda x: int(x.parent.name.replace("Part", ""))
    )

    files = [
        f for f in files
        if start <= int(f.parent.name.replace("Part", "")) <= end
    ]

    return files

### 8. Create CSV fil with features

Pick outputfolder 
Saves each participant and one file combined. 

Check for bad data 
- PPG & GSR: to many NaN (over 20 %)
- GSR: Flat signal (low std)
- GSR: Noisy (high std)
- PPG: Flat signal (low std)

*Output*: participant.csv and combined.csv 

In [ ]:
# cell 8

from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/Participants")
OUTPUT_DIR = Path("../data/processed/eda_ppg_features_for_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# SELECT PARTICIPANTS
participant_files = get_participant_files(DATA_DIR, 49, 50)

print("Participants loaded:", len(participant_files))


# Extract features
all_features = []
bad_participants = []

for file in participant_files:

    participant_id = file.parent.name
    print("\nProcessing:", participant_id)

    df = pd.read_csv(file)

    
    # SIGNAL QUALITY CHECK (GSR + PPG)
    gsr = pd.to_numeric(df["gsr"], errors="coerce")
    ppg = pd.to_numeric(df["ppg"], errors="coerce")

    # GSR checks
    if gsr.isna().mean() > 0.2:
        print("Too many NaNs in GSR")
        bad_participants.append(participant_id)
        continue

    if gsr.std() < 1e-3:
        print("Flat GSR signal")
        bad_participants.append(participant_id)
        continue

    if gsr.std() > 500:
        print("Extremely noisy GSR")
        bad_participants.append(participant_id)
        continue

    # PPG checks
    if ppg.isna().mean() > 0.2:
        print("Too many NaNs in PPG")
        bad_participants.append(participant_id)
        continue

    if ppg.std() < 1e-3:
        print("Flat PPG signal")
        bad_participants.append(participant_id)
        continue

  
    # FEATURE EXTRACTION
    features = extract_features_eda_ppg(df, participant_id)

    if features is None or len(features) == 0:
        print("No features extracted")
        bad_participants.append(participant_id)
        continue

    # SAVE INDIVIDUAL (CHANGE FILENAME HERE --> _features_test.csv")
    output_file = OUTPUT_DIR / f"{participant_id}_features_test.csv"
    features.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")

    all_features.append(features)


# Combine ALL
if len(all_features) > 0:
    features_df = pd.concat(all_features, ignore_index=True)

    
    # SAVE GLOBAL (CHANGE FILENAME HERE --> "features_all_test.csv" )
    features_df.to_csv(OUTPUT_DIR / "features_all_test.csv", index=False)

    print("\nDONE ")
    print("Shape:", features_df.shape)
else:
    print("\n No valid features extracted")


# SUMMARY
print("\nBad participants skipped:", len(bad_participants))
print(bad_participants)

Participants loaded: 2

Processing: Part49
Phasic stats:
min: 0.0
max: 55.36567407147446
std: 4.891353636452337
[Part49] Phasic peaks detected: 198
[Part49] Threshold: 0.05
[Part49] SCR count: 198
[Part49] Filtered SCR count: 198
✅ Saved: ..\data\processed\eda_ppg_features_for_test\Part49_features_test.csv

Processing: Part50
Phasic stats:
min: 0.0
max: 23.503800954699344
std: 2.4218560863014096
[Part50] Phasic peaks detected: 148
[Part50] Threshold: 0.05
[Part50] SCR count: 148
[Part50] Filtered SCR count: 147
✅ Saved: ..\data\processed\eda_ppg_features_for_test\Part50_features_test.csv

DONE ✅
Shape: (302, 16)

Bad participants skipped: 0
[]


In [13]:
features_df.describe()

,HR,HRV_RMSSD,RR_diff_std,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_quality,RR_valid,time_sec,SC_PH,SC_RR,EDA_slope
count,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.0,302.000000,302.000000,302.000000,302.000000
mean,73.200349,43.987554,43.867556,52.449251,822.841622,52.086838,679.983444,949.453642,73.182119,0.979522,1.0,1125.000000,17.288783,0.073731,0.000064
std,4.678858,8.397719,8.340205,19.998718,50.157282,19.859210,64.501022,83.927370,4.675867,0.026467,0.0,654.920045,22.937721,0.035162,0.000214
min,64.613831,23.339598,23.336260,18.341384,634.789474,18.221895,555.000000,720.000000,64.000000,0.857143,1.0,0.000000,0.030426,0.000000,-0.000285
25%,69.776135,38.396810,38.383008,35.683444,792.434211,35.436146,632.500000,876.250000,70.000000,0.969810,1.0,558.750000,3.241443,0.050000,-0.000155
50%,73.449887,43.963562,43.827575,50.521887,816.883562,50.183393,670.000000,950.000000,73.000000,0.986301,1.0,1125.000000,8.628886,0.066667,0.000158
75%,75.716066,49.464382,49.348100,65.529152,859.892857,65.041856,725.000000,1010.000000,76.000000,1.000000,1.0,1691.250000,22.991658,0.100000,0.000276
max,94.519526,68.641961,68.641249,114.326172,928.593750,113.571536,845.000000,1140.000000,95.000000,1.000000,1.0,2250.000000,137.359732,0.183333,0.000289
